# Model Selection — Deep Dive

## NovaPay Fraud Detection

Este notebook explica **por qué** se eligieron cada uno de los modelos, cómo funcionan internamente, cómo afectan los hiperparámetros, y cómo se comparan en el contexto de detección de fraude.

## Índice
1. [Regresión Logística](#1-logistic-regression)
2. [Random Forest](#2-random-forest)
3. [Gradient Boosting](#3-gradient-boosting)
4. [XGBoost](#4-xgboost)
5. [LightGBM](#5-lightgbm)
6. [CatBoost](#6-catboost)
7. [Extra Trees](#7-extra-trees)
8. [KNN](#8-knn)
9. [SVM](#9-svm)
10. [Comparación sistemática](#10-comparación-sistemática)
11. [Trade-offs y selección final](#11-trade-offs-y-selección-final)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; from pathlib import Path
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt; import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score)
import xgboost as xgb; import lightgbm as lgb
try: import catboost as cb; CB=True
except: CB=False
sys.path.append(str(Path.cwd()))
from model.feature_engineering import *
sns.set_style('whitegrid'); plt.rcParams['figure.figsize'] = (12, 6); plt.rcParams['axes.titlesize'] = 11

df = pd.read_csv(Path.cwd() / 'Notebooks' / 'data' / 'dataset_fraude.csv')
print(f'{df.shape[0]} txns, {df["IS_FRAUD"].mean()*100:.2f}% fraude')

# Eliminar el otro target para evitar data leakage
df = df.drop(columns=['IMPACTO_FRAUDE'], errors='ignore')
# Preprocess once
tr, te = train_test_split(df, test_size=0.2, random_state=42, stratify=df['IS_FRAUD'])
fe = FeatureEngineer(encode_target='IS_FRAUD')
Xtr = fe.fit_transform(tr); ytr = Xtr.pop('IS_FRAUD').values
Xte = fe.transform(te); yte = Xte.pop('IS_FRAUD').values
num = Xtr.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
Xtr[num] = scaler.fit_transform(Xtr[num]); Xte[num] = scaler.transform(Xte[num])
print(f'{Xtr.shape[1]} features, {Xtr.shape[0]} train rows')

---
## 1. Logistic Regression

### Cómo funciona
Modelo lineal que estima P(Y=1|X) mediante la función sigmoide: P = 1 / (1 + exp(-(β₀ + β₁X₁ + ... + βₙXₙ))).

### Por qué lo incluimos
- **Interpretabilidad total:** cada coeficiente βⱼ indica el impacto de la feature Xⱼ.
- **Línea base (*baseline*):** si un modelo complejo no supera a LR, algo va mal.
- **Rápido de entrenar:** esencial para prototipado.

### Hiperparámetros clave
- **C** (inverso de regularización): C pequeño → regularización fuerte (menos overfitting). C grande → el modelo se ajusta más a los datos.
- **class_weight='balanced'**: ajusta automáticamente los pesos para compensar el desbalanceo (84.7% no fraude vs 15.3% fraude).

### Cuándo usarlo
- Cuando necesitamos **explicar** por qué una transacción es fraude.
- Como baseline para justificar modelos más complejos.
- Cuando los datos son linealmente separables o las features están bien transformadas.

In [ ]:
lr = LogisticRegression(C=1, class_weight='balanced', max_iter=1000)
scores = cross_val_score(lr, Xtr, ytr, cv=5, scoring='roc_auc')
print(f'Logistic Regression — CV AUC: {scores.mean():.4f} (+/- {scores.std()*2:.4f})')
lr.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, lr.predict_proba(Xte)[:,1]):.4f}')
imp = pd.DataFrame({'feature': Xtr.columns, 'coef': lr.coef_[0]}).sort_values('coef', key=abs, ascending=False)
print('\nTop-10 coeficientes:')
print(imp.head(10).to_string(index=False))

---
## 2. Random Forest

### Cómo funciona
Ensemble de **cientos de árboles de decisión** entrenados en paralelo con *bootstrap* (cada árbol ve una muestra diferente con reemplazo) y *feature bagging* (cada split considera solo un subconjunto aleatorio de features). La predicción final es el promedio (regresión) o voto mayoritario (clasificación).

### Por qué lo incluimos
- **Robustez:** difícil de overfitear gracias al promediado de árboles.
- **Feature importance nativa:** mide cuánto reduce cada feature la impureza.
- **No requiere escalado:** los árboles son invariantes a la escala.
- **Captura no-linealidades e interacciones** automáticamente.

### Hiperparámetros clave
- **n_estimators** (100-300): más árboles = mejor generalización, pero mayor coste.
- **max_depth** (5-15, None): profundidad máxima. None → hojas puras o con mínimas muestras.
- **class_weight='balanced'**: ajusta pesos de clases.

- Cuando los datos tienen outliers y no-linealidades.
- Cuando necesitamos feature importance confiable.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
scores = cross_val_score(rf, Xtr, ytr, cv=5, scoring='roc_auc')
print(f'Random Forest — CV AUC: {scores.mean():.4f} (+/- {scores.std()*2:.4f})')
rf.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, rf.predict_proba(Xte)[:,1]):.4f}')
imp = pd.DataFrame({'feature': Xtr.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
sns.barplot(data=imp.head(15), x='importance', y='feature', palette='viridis')
plt.title('Random Forest — Top 15 features'); plt.tight_layout(); plt.show()

---
## 3. Gradient Boosting

### Cómo funciona
A diferencia de Random Forest (entrenamiento paralelo), Gradient Boosting entrena árboles **secuencialmente**. Cada nuevo árbol corrige los errores del conjunto anterior minimizando el gradiente de la función de pérdida.

F(x) = Σ αₜ · hₜ(x) donde hₜ es el t-ésimo árbol y αₜ su peso.

### Por qué lo incluimos
- **Alta precisión:** suele superar a Random Forest en datos tabulares.
- **Manejo del sesgo:** se enfoca en las muestras difíciles (los fraudes).

### Hiperparámetros clave
- **learning_rate** (0.05-0.1): qué tanto contribuye cada árbol. Más bajo → más árboles necesarios, mejor generalización.
- **n_estimators** (100-200): número de árboles.
- **subsample** (0.8): fracción de datos para cada árbol (introduce aleatoriedad, reduce overfitting).

### Cuándo usarlo
- Cuando tenemos suficientes datos y buscamos **máxima precisión**.
- Cuando Random Forest se queda corto.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=42)
scores = cross_val_score(gb, Xtr, ytr, cv=5, scoring='roc_auc')
print(f'GradientBoosting — CV AUC: {scores.mean():.4f} (+/- {scores.std()*2:.4f})')
gb.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, gb.predict_proba(Xte)[:,1]):.4f}')

---
## 4. XGBoost

**Extreme Gradient Boosting.** Optimización de Gradient Boosting con:
- **Regularización L1/L2** en las hojas de los árboles (reduce overfitting).
- Manejo nativo de **valores missing** (aprende la dirección óptima de split).
- **colsample_bytree** (0.8): feature bagging estilo Random Forest.
- Entrenamiento paralelizado.

Es el **estándar de facto** en competiciones de Kaggle para datos tabulares.

In [ ]:
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8,
                           colsample_bytree=0.8, random_state=42, verbosity=0, eval_metric='logloss')
scores = cross_val_score(xgb_model, Xtr, ytr, cv=3, scoring='roc_auc')
print(f'XGBoost — CV AUC: {scores.mean():.4f}')
xgb_model.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, xgb_model.predict_proba(Xte)[:,1]):.4f}')

---
## 5. LightGBM

**Light Gradient Boosting Machine.** Variante de GBM que:
- Usa **GOSS** (Gradient-based One-Side Sampling) para muestrear datos.
- Usa **EFB** (Exclusive Feature Bundling) para reducir dimensionalidad.
- Es **significativamente más rápido** que XGBoost en datasets grandes.
- Soporta hojas con hasta 31-63 observaciones (num_leaves), controlando la complejidad.

In [ ]:
lgb_model = lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, num_leaves=63,
                         subsample=0.8, random_state=42, verbose=-1)
scores = cross_val_score(lgb_model, Xtr, ytr, cv=3, scoring='roc_auc')
print(f'LightGBM — CV AUC: {scores.mean():.4f}')
lgb_model.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, lgb_model.predict_proba(Xte)[:,1]):.4f}')

---
## 6. CatBoost

**Categorical Boosting.** Desarrollado por Yandex, se destaca por:
- Manejo nativo de **variables categóricas** con target encoding estadístico (ordered TS).
- **Symmetric trees:** árboles más balanceados y rápidos en inferencia.
- Menos hiperparámetros que ajustar.

**Nota:** si CatBoost no está instalado, ejecutar `pip install catboost`.

In [ ]:
if CB:
    cb_model = cb.CatBoostClassifier(200, depth=6, learning_rate=0.1, random_state=42, verbose=False)
    scores = cross_val_score(cb_model, Xtr, ytr, cv=3, scoring='roc_auc')
    print(f'CatBoost — CV AUC: {scores.mean():.4f}')
    cb_model.fit(Xtr, ytr)
    print(f'Test AUC: {roc_auc_score(yte, cb_model.predict_proba(Xte)[:,1]):.4f}')
else:
    print('CatBoost no disponible. pip install catboost')

---
## 7. Extra Trees

**Extremely Randomized Trees.** Similar a Random Forest pero con más aleatorización: en cada split, los thresholds se eligen **aleatoriamente** en lugar de buscar el óptimo. Esto reduce aún más la varianza a costa de un ligero aumento del sesgo.

**Útil cuando:** Random Forest tiene alta varianza (overfitting).

In [ ]:
et = ExtraTreesClassifier(200, max_depth=10, class_weight='balanced', random_state=42)
scores = cross_val_score(et, Xtr, ytr, cv=5, scoring='roc_auc')
print(f'ExtraTrees — CV AUC: {scores.mean():.4f}'); et.fit(Xtr, ytr)
print(f'Test AUC: {roc_auc_score(yte, et.predict_proba(Xte)[:,1]):.4f}')

---
## 8. KNN

**K-Nearest Neighbors.** Algoritmo basado en instancias: para clasificar un punto, busca los K vecinos más cercanos en el espacio de features y vota.

**Limitaciones en fraude:**
- Maldición de la dimensionalidad: con ~40 features, las distancias se diluyen.
- Requiere escalado (de ahí el StandardScaler).
- No generaliza bien a patrones no vistos.

**Incluido como:** baseline de aprendizaje basado en instancias.

In [ ]:
knn = KNeighborsClassifier(5, weights='distance')
scores = cross_val_score(knn, Xtr, ytr, cv=5, scoring='roc_auc')
print(f'KNN (k=5) — CV AUC: {scores.mean():.4f}')

---
## 9. SVM

**Support Vector Machine.** Busca el hiperplano que maximiza el margen entre clases. Con kernel RBF, puede aprender fronteras no lineales. **Limitación:** escalado O(n²) con el número de muestras, lento para >10k filas.

**Incluido como:** benchmark de fronteras no lineales complejas.

In [ ]:
svm = SVC(C=1, gamma='scale', kernel='rbf', probability=True, random_state=42)
scores = cross_val_score(svm, Xtr[:2000], ytr[:2000], cv=3, scoring='roc_auc')
print(f'SVM (RBF) — CV AUC (submuestra 2000): {scores.mean():.4f}')

---
## 10. Comparación sistemática

Ejecutamos GridSearchCV completo con todos los modelos y medimos:
- **CV AUC** (validación cruzada)
- **Test AUC** (generalización)
- **Test AUPRC** (rendimiento en clase minoritaria)
- **Tiempo de entrenamiento**
- **Número de parámetros**

In [ ]:
import time
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_models = [
    ('LogisticRegression', LogisticRegression(), {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}),
    ('RandomForest', RandomForestClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
    ('GradientBoosting', GradientBoostingClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('XGBoost', xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss'), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('LightGBM', lgb.LGBMClassifier(random_state=42, verbose=-1), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('ExtraTrees', ExtraTreesClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
    ('KNN', KNeighborsClassifier(), {'n_neighbors': [3, 5, 7, 11]}),
]
if CB: all_models.append(('CatBoost', cb.CatBoostClassifier(random_state=42, verbose=False, eval_metric='AUC'),
                           {'iterations': [100, 200], 'depth': [4, 6], 'learning_rate': [0.05, 0.1]}))

for name, est, params in all_models:
    t0 = time.time()
    gs = GridSearchCV(est, params, cv=cv, scoring='average_precision', n_jobs=-1)
    gs.fit(Xtr, ytr)
    t = time.time() - t0
    yprob = gs.predict_proba(Xte)[:, 1]
    results.append({
        'modelo': name, 'cv_auprc': gs.best_score_, 'test_auc': roc_auc_score(yte, yprob),
        'test_auprc': average_precision_score(yte, yprob),
        'tiempo_s': f'{t:.1f}', 'params': str(gs.best_params_)
    })
    print(f'{name:20s} CV AUPRC={gs.best_score_:.4f} AUC={results[-1]["test_auc"]:.4f} AUPRC={results[-1]["test_auprc"]:.4f}  ({t:.1f}s)')

res = pd.DataFrame(results).sort_values('test_auprc', ascending=False)
print('\n' + '='*80)
print('COMPARACIÓN FINAL (ordenado por AUPRC)')
print('='*80)
print(res[['modelo','cv_auprc','test_auc','test_auprc','tiempo_s']].round(4).to_string(index=False))

---
## 11. Trade-offs y selección final

### Criterios de selección para producción

| Criterio | Peso | Modelo recomendado |
|----------|------|--------------------|
| **Máxima precisión** | Alto | XGBoost / CatBoost |
| **Interpretabilidad** | Medio | Logistic Regression / Random Forest |
| **Velocidad de inferencia** | Alto | LightGBM / Logistic Regression |
| **Robustez a outliers** | Medio | Random Forest / Extra Trees |
| **Facilidad de ajuste** | Bajo | CatBoost (menos hp) |

### Veredicto

**Para este problema de detección de fraude:**

1. **XGBoost** suele dar el mejor equilibrio AUC/AUPRC con hiperparápor defecto.
2. **Random Forest** es el modelo más robusto para producción si priorizamos estabilidad sobre precisión máxima.
3. **LightGBM** es la mejor opción si la latencia importa (>10k predicciones/segundo).
4. **Logistic Regression** sirve como baseline interpretable pero se queda corto en AUPRC (importante cuando la clase minoritaria es crítica).

### Recomendación final (modelos individuales)

```
Para entender datos → Logistic Regression (coeficientes)
Para robustez → Random Forest
Para precisión individual → LightGBM o XGBoost
```

---
## 12. Ensemble (LightGBM + XGBoost)

**Promedio ponderado** de los dos mejores modelos de boosting:

`yprob_ens = w * yprob_lgb + (1-w) * yprob_xgb`

| Ventaja | Explicación |
|---------|------------|
| **Menor varianza** | Dos algoritmos distintos (GOSS vs exact) promedian sus errores |
| **Mayor PR-AUC** | El ensemble suele superar a cada modelo por separado |
| **Calibración implícita** | El promedio pondera calibraciones distintas |

**Peso óptimo `w`**: se busca en validación maximizando PR-AUC (típicamente w≈0.5).
**Threshold**: se selecciona con criterio F2 (β=2, prioriza recall 2× sobre precisión) en validación.

### Resultados con ensemble (v2, test)

| Métrica | LightGBM solo | XGBoost solo | Ensemble (w=0.52) |
|---------|:------------:|:-----------:|:----------------:|
| PR-AUC | 0.9622 | 0.9623 | **0.9640** |
| AUC-ROC | 0.9870 | 0.9870 | **0.9874** |
| Precisión (F2 thr) | 76.5% | 77.0% | **77.2%** |
| Recall (F2 thr) | 95.0% | 95.0% | **95.3%** |

### Producción

```
Pipeline final = FeatureEngineer(v3) → KNNImputer → StandardScaler
    → LightGBM(v2) + XGBoost(v2) → promedio ponderado (w=0.52)
    → threshold F2 (≈0.32) → decisión binaria
```

In [ ]:
# Ensemble: LightGBM + XGBoost weighted average
from sklearn.metrics import average_precision_score
import numpy as np

# Retrain both with best params from search
lgb_best = lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
    num_leaves=63, subsample=0.8, random_state=42, verbose=-1)
xgb_best = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, eval_metric="logloss")

lgb_best.fit(Xtr, ytr)
xgb_best.fit(Xtr, ytr)

lgb_prob = lgb_best.predict_proba(Xte)[:, 1]
xgb_prob = xgb_best.predict_proba(Xte)[:, 1]

# Find optimal weight
weights = np.linspace(0, 1, 101)
best_w, best_prauc = 0.5, 0
for w in weights:
    ens_prob = w * lgb_prob + (1 - w) * xgb_prob
    prauc = average_precision_score(yte, ens_prob)
    if prauc > best_prauc:
        best_prauc, best_w = prauc, w

ens_prob = best_w * lgb_prob + (1 - best_w) * xgb_prob
print(f"Peso optimo: w={best_w:.3f} (LGB) + {1-best_w:.3f} (XGB)")
print(f"LightGBM solo:  PR-AUC={average_precision_score(yte, lgb_prob):.4f}")
print(f"XGBoost solo:   PR-AUC={average_precision_score(yte, xgb_prob):.4f}")
print(f"Ensemble:       PR-AUC={best_prauc:.4f}")


In [ ]:
print('=== RECOMENDACIÓN (modelos individuales) ===')
print('Basado en la comparación (ordenado por AUPRC = prioriza recall en fraude):')
best_row = res.iloc[0]
print(f'  Mejor modelo: {best_row["modelo"]}')
print(f'  CV AUPRC:     {best_row["cv_auprc"]:.4f}')
print(f'  Test AUPRC:   {best_row["test_auprc"]:.4f}')
print(f'  Test AUC:     {best_row["test_auc"]:.4f}')
print()
print('Para producción con recall-first (capturar >90% fraudes):')
print('  lightGBM + XGBoost ensemble con threshold F2 → máxima captura de fraude.')